# FIUBA - Maestría en Inteligencia Artificial
## Algoritmos Evolutivos 1 (2025)
### Desafío Práctico: Optimización de Ubicación de Antenas mediante PSO

**Alumno:** Martín Rodrigo Andujar  
**Docente:** Esp. Ing. Miguel Augusto Azar  

Este notebook resuelve el problema de encontrar la ubicación óptima de 3 antenas en un plano continuo de 100x100 km para maximizar la cobertura de 30 clientes distribuidos aleatoriamente. Incluye la curva de convergencia obligatoria y el diagrama de caja opcional solicitado por la cátedra.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. CONFIGURACIÓN DEL ESCENARIO
np.random.seed(42)
N_CLIENTES = 30
CLIENTES_X = np.random.uniform(0, 100, N_CLIENTES)
CLIENTES_Y = np.random.uniform(0, 100, N_CLIENTES)

N_ANTENAS = 3
RADIO_COBERTURA = 25.0
DIMENSION = N_ANTENAS * 2
X_MIN, X_MAX = 0.0, 100.0

def evaluar_fitness(particula):
    pos_antenas = particula.reshape(N_ANTENAS, 2)
    clientes_cubiertos = 0
    for i in range(N_CLIENTES):
        cx, cy = CLIENTES_X[i], CLIENTES_Y[i]
        for j in range(N_ANTENAS):
            ax, ay = pos_antenas[j][0], pos_antenas[j][1]
            if np.sqrt((cx - ax)**2 + (cy - ay)**2) <= RADIO_COBERTURA:
                clientes_cubiertos += 1
                break
    return -clientes_cubiertos

### 2. Ejecución del Algoritmo PSO e Historial para Diagrama de Caja
Para construir el diagrama de caja solicitado de forma opcional por la cátedra, ejecutaremos el algoritmo un total de 30 veces independientes para evaluar la estabilidad de la aptitud final alcanzada.

In [ ]:
N_PARTICULAS = 20
MAX_ITERACIONES = 50
W, C1, C2 = 0.5, 1.5, 1.5

ejecuciones_totales = 30
mejores_fitness_finales = []
historial_convergencia_ejemplo = []
mejor_pos_global = None
mejor_fitness_global = float('inf')

for ejecucion in range(ejecuciones_totales):
    posiciones = np.random.uniform(X_MIN, X_MAX, (N_PARTICULAS, DIMENSION))
    velocidades = np.random.uniform(-5, 5, (N_PARTICULAS, DIMENSION))
    
    p_best_pos = np.copy(posiciones)
    p_best_fitness = np.array([evaluar_fitness(p) for p in posiciones])
    
    g_best_idx = np.argmin(p_best_fitness)
    g_best_pos = np.copy(p_best_pos[g_best_idx])
    g_best_fitness = p_best_fitness[g_best_idx]
    
    convergencia_local = [abs(g_best_fitness)]
    
    for iteracion in range(MAX_ITERACIONES):
        for i in range(N_PARTICULAS):
            r1, r2 = np.random.rand(DIMENSION), np.random.rand(DIMENSION)
            velocidades[i] = (W * velocidades[i] + 
                              C1 * r1 * (p_best_pos[i] - posiciones[i]) + 
                              C2 * r2 * (g_best_pos - posiciones[i]))
            posiciones[i] = np.clip(posiciones[i] + velocidades[i], X_MIN, X_MAX)
            
            fit = evaluar_fitness(posiciones[i])
            if fit < p_best_fitness[i]:
                p_best_fitness[i] = fit
                p_best_pos[i] = np.copy(posiciones[i])
                if fit < g_best_fitness:
                    g_best_fitness = fit
                    g_best_pos = np.copy(posiciones[i])
                    
        convergencia_local.append(abs(g_best_fitness))
    
    mejores_fitness_finales.append(abs(g_best_fitness))
    if g_best_fitness < mejor_fitness_global:
        mejor_fitness_global = g_best_fitness
        mejor_pos_global = g_best_pos
        historial_convergencia_ejemplo = convergencia_local

print(f"Pruebas completadas. Mejor cobertura absoluta: {abs(mejor_fitness_global)} clientes.")

### 3. Visualización Completa (Convergencia, Cobertura y Diagrama de Caja)

In [ ]:
plt.figure(figsize=(18, 5))

# Gráfico 1: Curva de Convergencia (Obligatorio)
plt.subplot(1, 3, 1)
plt.plot(historial_convergencia_ejemplo, color='blue', linewidth=2, marker='o', markersize=3)
plt.title('Curva de Convergencia PSO')
plt.xlabel('Iteración')
plt.ylabel('Clientes Cubiertos')
plt.grid(True)

# Gráfico 2: Diagrama de Caja (Opcional solicitado)
plt.subplot(1, 3, 2)
plt.boxplot(mejores_fitness_finales, patch_artist=True, boxprops=dict(facecolor='lightblue'))
plt.title('Distribución de Fitness (30 Ejecuciones)')
plt.ylabel('Clientes Cubiertos al Final')
plt.xticks([1], ['PSO Continuo'])
plt.grid(True)

# Gráfico 3: Distribución Geométrica de Antenas
plt.subplot(1, 3, 3)
plt.scatter(CLIENTES_X, CLIENTES_Y, color='gray', label='Clientes', alpha=0.6)
antenas_optimas = mejor_pos_global.reshape(N_ANTENAS, 2)
for idx, antena in enumerate(antenas_optimas):
    plt.scatter(antena[0], antena[1], color='red', marker='^', s=120, label='Antena' if idx==0 else "")
    circulo = plt.Circle((antena[0], antena[1]), RADIO_COBERTURA, color='red', fill=True, alpha=0.12, linestyle='--')
    plt.gca().add_patch(circulo)
plt.title('Mapa de Cobertura Óptima Final')
plt.xlim(-10, 110) 
plt.ylim(-10, 110)
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()